In [1]:
import pickle
import pandas as pd

with open("model.bin", "rb") as f:
    dv, model = pickle.load(f)

df = pd.read_parquet(
    "data/yellow_tripdata_2023-03.parquet"
)

df["duration"] = (
    df.tpep_dropoff_datetime
    - df.tpep_pickup_datetime
).dt.total_seconds() / 60

df = df[
    (df.duration >= 1)
    & (df.duration <= 60)
].copy()

categorical = ["PULocationID", "DOLocationID"]

df[categorical] = (
    df[categorical]
    .fillna(-1)
    .astype(int)
    .astype(str)
)

X = dv.transform(
    df[categorical].to_dict(orient="records")
)

predictions = model.predict(X)

print("Rows:", len(predictions))
print("Mean:", predictions.mean())
print("Standard deviation:", predictions.std())

/home/anton/work/mlops/.venv/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DictVectorizer from version 1.5.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/anton/work/mlops/.venv/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LinearRegression from version 1.5.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Rows: 3316216
Mean: 14.203865642696083
Standard deviation: 6.247488852238703


In [2]:

from pathlib import Path
import pandas as pd

year = 2023
month = 3

assert len(df) == len(predictions)

df["ride_id"] = (
    f"{year:04d}/{month:02d}_"
    + df.index.astype(str)
)

df_result = pd.DataFrame({
    "ride_id": df["ride_id"],
    "predicted_duration": predictions,
})

output_file = Path("output") / "yellow_tripdata_2023-03_predictions.parquet"
output_file.parent.mkdir(parents=True, exist_ok=True)

df_result.to_parquet(
    output_file,
    engine="pyarrow",
    compression=None,
    index=False,
)

size_bytes = output_file.stat().st_size

print("Rows:", len(df_result))
print("Columns:", df_result.columns.tolist())
print(f"File size: {size_bytes / 1_000_000:.2f} MB")
print(df_result.head())


Rows: 3316216
Columns: ['ride_id', 'predicted_duration']
File size: 68.64 MB
     ride_id  predicted_duration
0  2023/03_0           16.245906
1  2023/03_1           26.134796
2  2023/03_2           11.884264
3  2023/03_3           11.997720
4  2023/03_4           10.234486


In [3]:

saved = pd.read_parquet(output_file)

assert len(saved) == len(predictions)
assert saved.columns.tolist() == [
    "ride_id",
    "predicted_duration",
]
assert saved["ride_id"].is_unique

print("Verified rows:", len(saved))
print("Mean prediction:", saved["predicted_duration"].mean())
print("File size (MB):", size_bytes / 1_000_000)


Verified rows: 3316216
Mean prediction: 14.203865642696083
File size (MB): 68.640926


In [4]:
import json

with open("Pipfile.lock") as f:
    lock = json.load(f)

print(lock["default"]["scikit-learn"]["hashes"][0])

FileNotFoundError: [Errno 2] No such file or directory: 'Pipfile.lock'